# 02 - OSM-only demo: is OSM good enough for neighborhood similarity?

Non-learned baseline: per-cell OSM POI counts + street stats, z-scored per city, cosine nearest neighbors.
One query per anchor (no averaging). Scope: shopping/dining/street character; parks and open space are out of scope. Runs locally or on Kaggle. Downloads are cached to `CACHE`.

**Before trusting results:** verify anchor coordinates on a map and write down your gut-check
expectations for Austin *before* looking at the output.

In [ ]:
# Kaggle/Colab: uncomment. Locally: pip install in your venv instead.
# %pip install -q "osmnx>=2.0" geopandas pyarrow scikit-learn

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import osmnx as ox
from scipy.spatial import cKDTree
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
warnings.filterwarnings("ignore")

ox.settings.use_cache = True
CACHE = Path("/kaggle/working/cache") if os.path.exists("/kaggle") else Path("../data/interim")
CACHE.mkdir(parents=True, exist_ok=True)

# Search area = OSM administrative boundary of each place (fetched in the next section).
PLACES = {
    "houston": "Houston, Texas, USA",
    "austin":  "Austin, Texas, USA",
    "seattle": "Seattle, Washington, USA",
    "atlanta": "Atlanta, Georgia, USA",
}
# Optional: intersect a city's boundary with a box (west, south, east, north) to shrink huge areas
# (Houston's city limits are very large). Example: CLIP = {"houston": (-95.80, 29.55, -95.05, 30.00)}
CLIP = {}
# Approximate - VERIFY ON A MAP. (lat, lon)
ANCHORS = {
    "Rice Village":    ("houston", 29.7175, -95.4170),
    "Montrose":        ("houston", 29.7408, -95.3908),
    "Chinatown (Bellaire)": ("houston", 29.7055, -95.5370),
}
SPACING_M = 400     # grid spacing
RADIUS_M  = 800     # walkshed radius per cell
NMS_M     = 1600    # min separation between returned results
TAG_KEYS  = ["amenity", "shop", "leisure", "tourism", "office", "craft", "healthcare"]
MIN_TAG_COUNT = 30  # drop rare tags from vocab

# --- similarity tuning ---
Z_CLIP = 3.0                 # clip z-scores so rare tags cannot dominate
W_POI, W_STREET = 1.0, 1.0   # total influence of each feature group
# Leisure focus: exclude whole keys and utility/civic/medical tags (weight 0 = ignored in similarity).
EXCLUDE_KEYS = {"healthcare", "office"}
EXCLUDE_TAGS = {
    # parking / vehicles / money
    "amenity=parking", "amenity=parking_space", "amenity=parking_entrance", "amenity=fuel",
    "amenity=car_wash", "amenity=car_rental", "amenity=charging_station", "amenity=atm", "amenity=bank",
    # street furniture / utilities
    "amenity=toilets", "amenity=drinking_water", "amenity=bench", "amenity=waste_basket",
    "amenity=waste_disposal", "amenity=recycling", "amenity=vending_machine", "amenity=post_box",
    "amenity=bicycle_parking", "amenity=bicycle_repair_station", "amenity=shelter",
    # civic / medical / education
    "amenity=school", "amenity=kindergarten", "amenity=college", "amenity=university",
    "amenity=doctors", "amenity=dentist", "amenity=clinic", "amenity=hospital", "amenity=pharmacy",
    "amenity=police", "amenity=fire_station", "amenity=post_office", "amenity=townhall",
    "amenity=courthouse", "amenity=social_facility", "amenity=driving_school", "amenity=place_of_worship",
}
def tag_weight(t):
    if t.split("=")[0] in EXCLUDE_KEYS or t in EXCLUDE_TAGS: return 0.0
    return 1.0

import hashlib, json
def cfg_hash(*parts):
    return hashlib.md5(json.dumps(parts, default=str).encode()).hexdigest()[:8]
def city_crs(bbox):
    lon = (bbox[0] + bbox[2]) / 2
    return f"EPSG:{32600 + int((lon + 180) // 6) + 1}"   # UTM north

## 0. City boundaries from OSM

In [ ]:
from shapely.geometry import box
import shapely

def load_boundary(city, query):
    f = CACHE / f"{city}_boundary.geojson"
    if f.exists():
        g = gpd.read_file(f)
    else:
        g = ox.geocode_to_gdf(query)
        g[["geometry"]].to_file(f, driver="GeoJSON")
    poly = g.geometry.iloc[0]
    assert poly.geom_type in ("Polygon", "MultiPolygon"), f"{query}: got {poly.geom_type}, not a boundary"
    if city in CLIP:
        poly = poly.intersection(box(*CLIP[city]))
    return poly

BOUNDARY = {c: load_boundary(c, q) for c, q in PLACES.items()}
CITIES = {c: p.bounds for c, p in BOUNDARY.items()}   # (west, south, east, north), derived
for c, b in CITIES.items():
    print(c, [round(v, 3) for v in b], "| area km2 ~", round(gpd.GeoSeries([BOUNDARY[c]], crs=4326).to_crs(city_crs(b)).area.iloc[0] / 1e6))

## 1. Download (cached)

In [ ]:
def load_pois(city, poly):
    bbox = poly.bounds
    f = CACHE / f"{city}_pois_{cfg_hash(bbox, round(poly.area, 8), TAG_KEYS)}.parquet"   # new area/tags -> new file
    if f.exists():
        return pd.read_parquet(f)
    g = ox.features_from_polygon(poly, tags={k: True for k in TAG_KEYS})
    g = g.to_crs(city_crs(bbox))
    rows = []
    cen = g.geometry.centroid
    for k in TAG_KEYS:
        if k not in g: continue
        m = g[k].notna()
        rows.append(pd.DataFrame({"x": cen[m].x.values, "y": cen[m].y.values,
                                  "tag": k + "=" + g.loc[m, k].astype(str).values}))
    df = pd.concat(rows, ignore_index=True)
    df.to_parquet(f)
    return df

def load_streets(city, poly):
    bbox = poly.bounds
    h = cfg_hash(bbox, round(poly.area, 8))
    f = CACHE / f"{city}_nodes_{h}.parquet"
    if f.exists():
        return pd.read_parquet(f), pd.read_parquet(CACHE / f"{city}_edges_{h}.parquet")
    G = ox.graph_from_polygon(poly, network_type="walk")
    G = ox.project_graph(G, to_crs=city_crs(bbox))
    nodes, edges = ox.graph_to_gdfs(G)
    n = pd.DataFrame({"x": nodes.geometry.x, "y": nodes.geometry.y, "deg": nodes["street_count"]})
    mid = edges.geometry.interpolate(0.5, normalized=True)
    e = pd.DataFrame({"x": mid.x.values, "y": mid.y.values, "len": edges["length"].values})
    n.to_parquet(f); e.to_parquet(CACHE / f"{city}_edges_{h}.parquet")
    return n, e

pois, nodes, edges, crs = {}, {}, {}, {}
for c, poly in BOUNDARY.items():
    pois[c] = load_pois(c, poly)
    nodes[c], edges[c] = load_streets(c, poly)
    crs[c] = city_crs(poly.bounds)
    print(c, len(pois[c]), "POIs |", len(nodes[c]), "nodes |", len(edges[c]), "edges |", crs[c])

## 2. Grid + per-cell features

In [ ]:
from pyproj import Transformer

def make_grid(city):
    w, s, e, n = CITIES[city]
    t = Transformer.from_crs("EPSG:4326", crs[city], always_xy=True)
    (x0, y0), (x1, y1) = t.transform(w, s), t.transform(e, n)
    xs = np.arange(x0, x1, SPACING_M); ys = np.arange(y0, y1, SPACING_M)
    X, Y = np.meshgrid(xs, ys)
    poly_xy = gpd.GeoSeries([BOUNDARY[city]], crs=4326).to_crs(crs[city]).iloc[0]
    inside = shapely.contains_xy(poly_xy, X.ravel(), Y.ravel())   # keep only cells inside the boundary
    return np.c_[X.ravel(), Y.ravel()][inside]

vocab_counts = pd.concat([p["tag"] for p in pois.values()]).value_counts()
VOCAB = vocab_counts[vocab_counts >= MIN_TAG_COUNT].index.tolist()
tag_id = {t: i for i, t in enumerate(VOCAB)}
print("vocab size:", len(VOCAB))
NAMES = VOCAB + ["intersection_density", "street_km_per_km2", "mean_edge_len"]
print(vocab_counts.head(40).to_string())

def featurize(city):
    centers = make_grid(city)
    p = pois[city]; p = p[p["tag"].isin(tag_id)]
    ptree = cKDTree(p[["x", "y"]].values); pids = p["tag"].map(tag_id).values
    nd = nodes[city]; ntree = cKDTree(nd[["x", "y"]].values); inter = (nd["deg"].values >= 3)
    ed = edges[city]; etree = cKDTree(ed[["x", "y"]].values); elen = ed["len"].values
    area_km2 = np.pi * (RADIUS_M / 1000) ** 2
    poi_nb = ptree.query_ball_point(centers, RADIUS_M)
    n_nb = ntree.query_ball_point(centers, RADIUS_M)
    e_nb = etree.query_ball_point(centers, RADIUS_M)
    F = np.zeros((len(centers), len(VOCAB)), dtype=np.float32)
    street = np.zeros((len(centers), 3), dtype=np.float32)
    for i in range(len(centers)):
        if poi_nb[i]:
            F[i] = np.bincount(pids[poi_nb[i]], minlength=len(VOCAB))
        ni = n_nb[i]; ei = e_nb[i]
        street[i] = [inter[ni].sum() / area_km2 if ni else 0,
                     elen[ei].sum() / 1000 / area_km2 if ei else 0,
                     elen[ei].mean() if ei else 0]
    X = np.c_[np.log1p(F), np.log1p(street[:, :2]), street[:, 2:]]
    # keep only cells with some urban signal
    TW = np.array([tag_weight(t) for t in VOCAB], dtype=np.float32)
    ok = ((F * TW).sum(1) >= 3) | (street[:, 0] > 20)   # count only leisure-relevant POIs
    return centers, X.astype(np.float32), ok

feat = {}
for c in CITIES:
    centers, X, ok = featurize(c)
    feat[c] = dict(centers=centers, X=X, ok=ok)
    print(c, "cells:", len(centers), "usable:", int(ok.sum()))

# per-city centering + scaling (city-leakage fix), fit on usable cells only
tw = np.array([tag_weight(t) for t in VOCAB], dtype=np.float32)
col_w = np.r_[tw * W_POI / np.sqrt((tw ** 2).sum()), np.full(3, W_STREET / np.sqrt(3))]  # equal group influence
SHOW = np.r_[tw > 0.5, np.ones(3, dtype=bool)]   # features eligible for the "shared" explanation

for c, d in feat.items():
    mu, sd = d["X"][d["ok"]].mean(0), d["X"][d["ok"]].std(0) + 1e-6
    d["Z"] = np.clip((d["X"] - mu) / sd, -Z_CLIP, Z_CLIP)
    Zw = d["Z"] * col_w
    d["Zn"] = Zw / (np.linalg.norm(Zw, axis=1, keepdims=True) + 1e-9)

## 3. City-leakage probe (raw vs per-city standardized)

In [ ]:
def probe(key):
    Xs = np.vstack([feat[c][key][feat[c]["ok"]] for c in feat])
    y = np.concatenate([[c] * int(feat[c]["ok"].sum()) for c in feat])
    return cross_val_score(LogisticRegression(max_iter=300), Xs, y, cv=3).mean()

print("city predictable from raw features :", round(probe("X"), 3))
print("city predictable after centering   :", round(probe("Z"), 3), "(closer to chance = less leakage)")

## 4. Query: one vector per anchor, NMS, place-name labels

In [ ]:
def to_xy(city, lat, lon):
    return Transformer.from_crs("EPSG:4326", crs[city], always_xy=True).transform(lon, lat)

def to_latlon(city, x, y):
    lon, lat = Transformer.from_crs(crs[city], "EPSG:4326", always_xy=True).transform(x, y)
    return lat, lon

def search(anchor, target_city, k=8):
    acity, lat, lon = ANCHORS[anchor]
    ax, ay = to_xy(acity, lat, lon)
    A = feat[acity]
    ai = np.argmin(((A["centers"] - [ax, ay]) ** 2).sum(1))
    q = A["Zn"][ai]
    T = feat[target_city]
    sims = T["Zn"] @ q
    sims[~T["ok"]] = -9
    if target_city == acity:  # exclude the anchor's own neighborhood
        sims[np.linalg.norm(T["centers"] - A["centers"][ai], axis=1) < NMS_M] = -9
    order = np.argsort(-sims); chosen = []
    for j in order:
        if sims[j] < -1: break
        if all(np.linalg.norm(T["centers"][j] - T["centers"][c]) >= NMS_M for c in chosen):
            chosen.append(j)
        if len(chosen) == k: break
    names = np.array(NAMES)
    out = []
    for j in chosen:
        lat2, lon2 = to_latlon(target_city, *T["centers"][j])
        shared = np.where(SHOW, np.minimum(A["Z"][ai], T["Z"][j]), -9); top = names[np.argsort(-shared)[:4]]
        out.append(dict(sim=round(float(sims[j]), 3), lat=round(lat2, 5), lon=round(lon2, 5),
                        map=f"https://maps.google.com/?q={lat2:.5f},{lon2:.5f}", shared=", ".join(top)))
    return pd.DataFrame(out)

pd.set_option("display.max_colwidth", 80, "display.width", 200)
for a in ANCHORS:
    for tc in CITIES:
        print(f"\n=== {a}  ->  {tc} ===")
        print(search(a, tc).to_string(index=False))

## 5. How to judge it
- Within-Houston: do each anchor's nearest Houston cells look like the same kind of place?
- Austin: compare against the gut-check list you wrote *before* running.
- If results are dominated by generic density (e.g. everything downtown-ish), OSM POIs alone are too coarse -> add reviews/text.
- If the top shared features are sensible (restaurants/cafes/boutiques for Rice Village, parks/trails for Memorial Park), OSM is a usable base signal.